# Implicit Q-Learning on HalfCheetah-v5

**Person 5 — theory, numerical example, notebook narrative.**
**Branch:** `feature/notebook-presentation` · **Issue:** [#4](https://github.com/MedvAx-AI/iql-halfcheetah/issues/4)

This notebook explains offline RL and IQL, works one transition by hand, and imports shared code from `iql_project`. It does **not** re-implement the learner in cells.

| Status | Note |
|---|---|
| Narrative + manual example | Ready |
| Environment / dataset APIs | Merged from Person 2 (`main`) |
| IQL / train / eval / video | Depend on Persons 3–4 |
| Measured results + Colab acceptance | After M2–M3; Person 1 verifies |

Shared contracts: `IQL_5_Person_Project_Plan.md`, `docs/INTERFACES.md`, `configs/halfcheetah.toml`, `docs/ENVIRONMENT_DATASET.md`.


## 1. Introduction — offline RL and project goal

### Online vs offline

In **online** RL the agent interacts with the environment and can explore.

In **offline** RL the agent receives a **fixed dataset** of transitions

$$(s, a, r, s', \text{terminated}, \text{truncated})$$

collected beforehand. During training it cannot query the environment for new actions. Standard Q-learning is then dangerous: a $\max$ over actions can exploit out-of-distribution actions that look artificially good.

### What this project demonstrates

1. Load Minari `mujoco/halfcheetah/medium-v0`.
2. Train **Implicit Q-Learning (IQL)** on that fixed data only.
3. Evaluate in Gymnasium **HalfCheetah-v5**.
4. Show a demo video and report raw returns.

Roles: Person 2 data/env, Person 3 IQL/train, Person 4 evaluation/video, Person 5 explanation/slides, Person 1 integration/Colab.


## 2. Setup — package imports (no duplicated learner)

Install from the repo root with the locked Python 3.11 environment:

```text
uv sync --frozen --extra dev --extra notebook
uv run --frozen jupyter lab notebooks/iql_halfcheetah.ipynb
```

Optional dataset download (Person 2 path; large, explicit only):

```text
uv run --frozen python scripts/check_dataset.py --download
```

If you use a custom Minari cache, set `MINARI_DATASETS_PATH` in this process too.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [ ]:
from iql_project import __version__
from iql_project.cli import main
from iql_project.config import ProjectConfig, load_config

print("Package version:", __version__)
print("Repo root:", ROOT)
assert main(["check"]) == 0, "iql-project check failed"

config_path = ROOT / "configs" / "halfcheetah.toml"
config = load_config(config_path) if config_path.exists() else ProjectConfig()
print(config)
print(
    "Convention: beta := inverse_temperature =",
    config.inverse_temperature,
    "| expectile =",
    config.expectile,
    "| max_weight =",
    config.max_weight,
)

## 3. Environment — HalfCheetah-v5

HalfCheetah-v5 observes 17 continuous positions/velocities and controls six joint torques in `[-1, 1]`. The goal is forward motion with low control effort; the default reward is horizontal velocity minus $0.1$ times the squared action norm. The dataset records a 1,000-step horizon. Time-limit **truncations** end episodes while preserving the Q bootstrap.

Person 2 implements `make_evaluation_env(config, render_mode=None)` from the dataset collection EnvSpec (MuJoCo **3.2.3** matches collection; see `docs/SIMULATOR_COMPATIBILITY.md` and `docs/ENVIRONMENT_DATASET.md`).

**Note:** recovery loads the **local** Minari dataset (`download=False`). If the dataset is missing, the next cell skips instead of failing the whole notebook. Download explicitly before a full env demo.


In [ ]:
try:
    from iql_project.environment import make_evaluation_env

    env = make_evaluation_env(config, render_mode=None)
    print("observation_space:", env.observation_space)
    print("action_space:", env.action_space)
    obs, info = env.reset(seed=config.seed)
    print("reset obs shape:", getattr(obs, "shape", type(obs)))
    env.close()
except FileNotFoundError as exc:
    print("SKIP env demo: dataset not downloaded yet.")
    print("Download with: python scripts/check_dataset.py --download")
    print("Details:", exc)
except Exception as exc:
    print("SKIP env demo:", type(exc).__name__, "-", exc)

## 4. Offline dataset — Minari medium

`mujoco/halfcheetah/medium-v0` contains 1,000 episodes / 1,000,000 transitions. Person 2 validated: untransformed reward mean/std ≈ 12.09 / 4.70; episode return mean/std ≈ 12,089 / 3,009; every episode has 1,000 steps; zero true terminations and 1,000 truncations. These are **dataset** statistics, not trained-policy results.

Loader contract (`docs/INTERFACES.md`):

- float32 batches; observations `(B, 17)`, actions `(B, 6)`, rewards `(B, 1)`
- bool end flags `(B, 1)`
- **termination** disables Q bootstrap; **truncation alone does not**
- observation mean/std fitted on offline data only (std floored at `1e-3`)

`load_offline_dataset(config)` defaults to `download=False`.


In [ ]:
import numpy as np

dataset = None
try:
    from iql_project.dataset import load_offline_dataset

    dataset = load_offline_dataset(config, download=False)
    rng = np.random.default_rng(config.seed)
    batch = dataset.sample(config.batch_size, rng)
    print("transitions:", len(dataset))
    print("batch shapes:", batch.observations.shape, batch.actions.shape)
    print("info:", dataset.info)
    if hasattr(dataset, "metadata"):
        print("statistics:", dataset.metadata.get("statistics", dataset.metadata))
except FileNotFoundError as exc:
    print("SKIP dataset demo: local Minari cache missing.")
    print("Download with: python scripts/check_dataset.py --download")
    print("Details:", exc)
except Exception as exc:
    print("SKIP dataset demo:", type(exc).__name__, "-", exc)

## 5. What is IQL?

Paper: Kostrikov, Nair, Levine — *Offline Reinforcement Learning with Implicit Q-Learning* (ICLR 2022).

### Motivation

Offline Q-learning that takes $\max_{a'} Q(s', a')$ queries actions that may be absent from the dataset. Those OOD actions can receive inflated values and ruin the policy.

IQL instead:

1. Fits $V(s)$ to **dataset actions only**, via **expectile regression** toward upper expectiles of $Q(s,a)$.
2. Fits $Q(s,a)$ with a **V-backed** target (no max over unseen actions).
3. Extracts a policy by **advantage-weighted regression** on dataset actions.

### The three losses (project convention)

Let $\tau =$ `expectile` (default $0.7$) and $\beta =$ **`inverse_temperature`** (default $3.0$).
In this repository **beta is inverse temperature**, not temperature: larger $\beta$ concentrates more on high-advantage actions.

**1. Value / expectile loss**

$$
u = Q(s,a) - V(s), \qquad
w_\tau(u) = \begin{cases}\tau & u \ge 0 \\ 1-\tau & u < 0\end{cases}, \qquad
\mathcal{L}_V = \mathbb{E}\big[w_\tau(u)\, u^2\big]
$$

**2. Q loss with terminated mask**

$$
y = r + \gamma\,(1-\mathbb{1}_{\text{terminated}})\, V(s'), \qquad
\mathcal{L}_Q = \mathbb{E}\big[(Q(s,a) - y)^2\big]
$$

Truncation must **not** zero the bootstrap. Only true termination does.

**3. Policy loss with capped advantage weight**

$$
A(s,a) = Q(s,a) - V(s), \qquad
w = \min\big(\exp(\beta A),\; w_{\max}\big), \qquad
\mathcal{L}_\pi = \mathbb{E}\big[- w \log \pi_\theta(a\mid s)\big]
$$

with $w_{\max}=$ `max_weight` (default $100$).


## 6. Manual IQL update — one numerical transition

Pedagogy only (plain arithmetic), not a second learner.

| Symbol | Value | Meaning |
|---|---|---|
| $Q(s,a)$ | $8.0$ | current Q prediction |
| $V(s)$ | $5.0$ | current V at $s$ |
| $V(s')$ | $6.0$ | current V at next state |
| $r$ | $1.0$ | reward |
| $\gamma$ | $0.99$ | `config.discount` |
| $\tau$ | $0.7$ | expectile |
| $\beta$ | $3.0$ | **inverse temperature** |
| $w_{\max}$ | $100$ | policy weight cap |


In [ ]:
import math

Q_sa = 8.0
V_s = 5.0
V_s_next = 6.0
reward = 1.0
gamma = config.discount
tau = config.expectile
beta = config.inverse_temperature  # inverse temperature, NOT temperature
max_weight = config.max_weight

# 1) Expectile weight for V
u = Q_sa - V_s
expectile_weight = tau if u >= 0 else (1.0 - tau)
v_loss_term = expectile_weight * (u**2)

# 2) Masked Q targets
y_continue = reward + gamma * (1.0 - 0.0) * V_s_next  # terminated=False
y_terminal = reward + gamma * (1.0 - 1.0) * V_s_next  # terminated=True

# 3) Capped policy weight
advantage = Q_sa - V_s
raw_weight = math.exp(beta * advantage)
policy_weight = min(raw_weight, max_weight)

print("residual u = Q - V =", u)
print("expectile weight w_tau =", expectile_weight)
print("V loss term w_tau * u^2 =", v_loss_term)
print("Q target (unterminated) y =", y_continue)
print("Q target (terminated) y =", y_terminal)
print("advantage A =", advantage)
print("raw exp(beta * A) =", raw_weight)
print("capped policy weight =", policy_weight)
print("beta convention: inverse_temperature =", beta)

assert abs(v_loss_term - 6.3) < 1e-9
assert abs(y_continue - 6.94) < 1e-9
assert abs(y_terminal - 1.0) < 1e-9
assert policy_weight == max_weight
print("Manual example checks passed.")

### Reading the numbers

- Residual $u=3>0$ → expectile weight $\tau=0.7$. V is pulled toward this better-than-$V$ dataset action.
- Unterminated target $y=1+0.99\cdot 6=6.94$. If **terminated**, bootstrap is masked and $y=r=1$.
- Advantage $A=3$ with $\beta=3$ gives $\mathrm{e}^{9}\approx 8103$, then **capped** to $100$.

Bad-action contrast ($Q=3$, $V=5$): $A=-2$, $\mathrm{e}^{-6}\approx 0.0025$ → almost no imitation.


## 7. Package surface (Person 3) — agreed network factory

Public API is `build_networks(config, info) -> dict[str, nn.Module]` with keys `q1`, `q2`, `value`, `policy`, `target_q1` and `target_q2` (`docs/IQL.md`). Do **not** import a separate `QNetwork` / `VNetwork` / `PolicyNetwork` API from the notebook.

Losses, the tanh-Gaussian likelihood, checkpoint fields and resume are documented in `docs/IQL.md`. Evaluation reloads preprocessing with `dataset_info_from_checkpoint` before constructing `IQLAgent`.


In [ ]:
from iql_project.iql import IQLAgent
from iql_project.networks import build_networks

print("build_networks:", build_networks)
print("IQLAgent:", IQLAgent)

if dataset is None:
    print(
        "SKIP network build: dataset.info unavailable "
        "(download dataset first, or Person 3 pending)."
    )
else:
    try:
        nets = build_networks(config, dataset.info)
        print("networks:", sorted(nets))
    except NotImplementedError as exc:
        print("Pending Person 3:", exc)

## 8. Training protocol (Persons 3–4)

1. Load config + offline dataset (no env interaction for learning).
2. For `total_steps` updates: sample batch → `agent.update(batch)` → log JSONL.
3. Checkpoint under `checkpoints/<run_id>/`.
4. Write `results/<run_id>/manifest.json` with git commit, seeds, hardware, checksums.

Defaults: `batch_size=256`, `total_steps=500000`, `seed=0`, device `cpu` unless a CUDA lock is documented.


In [ ]:
from iql_project.train import train

print("train:", train)
print(
    "Run offline training with "
    "`iql-project train --config configs/halfcheetah.toml --run-dir results/<run_id>`. "
    "Resume with `--resume checkpoints/<run_id>/step_<update>.pt`. "
    "Only `total_steps` and `checkpoint_interval` may differ from the checkpoint."
)
print("This outline does not launch the default 500_000 updates.")

## 9. Results and evaluation (Person 4)

Evaluation uses the recovered Minari environment, saved checkpoint normalization and deterministic actions. Every checkpoint and the uniform random baseline use evaluation seeds **10000–10009**, separately for training seeds **0, 1, 2**.

Report raw environment returns. Within one run, error bars show population standard deviation across episodes (`ddof=0`). Across training seeds, report the mean and sample standard deviation (`ddof=1`) of the three run means. No D4RL normalized score is assumed.

Generate artifacts with `python scripts/run_experiments.py --steps 100000 --checkpoint-every 25000 --run-prefix person4_100k --video`, or evaluate existing checkpoints with `iql-project evaluate --checkpoint ... --run-dir ...`. See `docs/EVALUATION.md`.


In [ ]:
import json
from pathlib import Path

from IPython.display import Image, display

from iql_project.evaluate import evaluate, record_video
from iql_project.reporting import evaluate_checkpoint, summarize_runs

print("Evaluation API:", evaluate, record_video, evaluate_checkpoint, summarize_runs)
RESULTS_DIR = ROOT / "results/person4_100k_aggregate"
summary_path = RESULTS_DIR / "summary.json"
if summary_path.exists():
    measured = json.loads(summary_path.read_text())
    print("Protocol met:", measured["minimum_seed_episode_protocol_met"])
    print("Across training seeds (run means):")
    for checkpoint_result in measured["across_runs"]:
        print(checkpoint_result)
    display(Image(filename=str(RESULTS_DIR / "plots/evaluation_returns.png")))
else:
    print("No local measured artifacts at", RESULTS_DIR)
    print("Run the documented experiment or fetch the team's artifacts first.")

## 10. Demo, limitations, Colab acceptance

### Demo

A deterministic episode with the preselected seed 10000 is recorded for each final checkpoint in `videos/<run_id>/demo_seed_10000.mp4`. The example below plays seed 0 when the local artifact exists. Public artifact links/checksums belong in the final release handoff.

### Limitations (current)

- IQL updates, offline training, checkpoint/resume, evaluation, baseline, plots and MP4 recording are implemented.
- A 1,000-update smoke run checks the pipeline but does not establish a strong policy. Use the measured report and its actual update count.
- Dataset and artifact cells skip cleanly when their local files are absent.
- Fresh Colab `Run all` is Person 1 + Person 5 acceptance after integration.
- Headless Linux rendering may need `MUJOCO_GL=egl`.

### Tested commit / runtime (fill after Colab)

| Field | Value |
|---|---|
| git commit | _TBD after clean Colab Run all_ |
| config | `configs/halfcheetah.toml` |
| training seeds | `0, 1, 2` |
| evaluation seeds | `10000–10009` |
| runtime | _TBD_ |
| Python / platform | _TBD_ |


In [ ]:
from IPython.display import Video

VIDEO_PATH = ROOT / "videos/person4_100k_seed_0/demo_seed_10000.mp4"
if VIDEO_PATH.exists():
    display(Video(filename=str(VIDEO_PATH), embed=True))
else:
    print("No local demo at", VIDEO_PATH, "— record or fetch the artifact first.")

## 11. References

1. Ilya Kostrikov, Ashvin Nair, Sergey Levine. **Offline Reinforcement Learning with Implicit Q-Learning.** ICLR 2022. https://arxiv.org/abs/2110.06169
2. Official IQL implementation (cite commit if adapting): https://github.com/ikostrikov/implicit_q_learning
3. Minari `mujoco/halfcheetah/medium-v0`: https://minari.farama.org/datasets/mujoco/halfcheetah/medium-v0/
4. Gymnasium HalfCheetah: https://gymnasium.farama.org/environments/mujoco/half_cheetah/
5. Project docs: `docs/INTERFACES.md`, `docs/ENVIRONMENT_DATASET.md`, `IQL_5_Person_Project_Plan.md`

No third-party IQL code is vendored here. If Person 3 adapts snippets, record source commit and license in the PR.
